<style>
  .jp-MarkdownCell h1, .text_cell_render h1 { color: #312e81; font-weight: 700; border-bottom: 1px solid #6366f1; padding-bottom: 0.35em; }
  .jp-MarkdownCell h2, .text_cell_render h2 { color: #4338ca; font-weight: 600; margin-top: 1em; border-left: 3px solid #6366f1; padding-left: 0.4em; }
  .jp-MarkdownCell h3, .text_cell_render h3 { color: #0f766e; font-weight: 600; margin-top: 0.9em; border-left: 3px solid #14b8a6; padding-left: 0.35em; }
  .jp-MarkdownCell h4, .text_cell_render h4 { color: #374151; font-weight: 600; margin-top: 0.75em; border-left: 2px solid #94a3b8; padding-left: 0.35em; }
  .jp-MarkdownCell a, .text_cell_render a { color: #4f46e5; }
  .jp-MarkdownCell code, .text_cell_render code { background: #e0e7ff; color: #3730a3; padding: 0.15em 0.4em; border-radius: 4px; font-size: 0.9em; }
  .jp-MarkdownCell pre, .text_cell_render pre { background: #f8fafc; border: 1px solid #e2e8f0; border-left: 3px solid #6366f1; border-radius: 4px; padding: 0.6em 0.8em; }
  .jp-MarkdownCell strong, .text_cell_render strong { color: #1e293b; }
  .jp-MarkdownCell li::marker, .text_cell_render li::marker { color: #6366f1; }
</style>

# 6. Agentic Workflows with Coder and “Vibe Coding” (Instructor: Mohammad Sada)

The tutorial then shifts to agentic AI workflows using the Coder platform hosted at coder.nrp-nautilus.io. Participants will create isolated, persistent workspaces, which provide a sandboxed environment suitable for safe experimentation. Within these workspaces, attendees will explore the concept of “Vibe Coding,” where AI agents act as coders that can be managed programmatically through the Coder CLI.

Below we install the Coder CLI, log in to NRP's Coder instance, create a workspace from the **general-template**, install **Aider**, clone a repo, and run an agentic workflow: have Aider create a Python sudoku solver, then push a branch and open a merge request. LLM calls use the NRP endpoint (`OPENAI_API_BASE=https://ellm.nrp-nautilus.io/v1`). **API key:** Set `OPENAI_API_KEY` in your environment or obtain a token from the [LLM token page](https://nrp.ai/llmtoken); the notebook uses it for Aider, OpenCode, and Crush.

### Coder at a glance

[Coder](https://coder.nrp-nautilus.io) on NRP works like JupyterHub: sign in with your institution (OpenID Connect). Cluster admins must approve first-time access—if you don’t see access, ask in [Matrix](https://nrp.ai/contact/). You get up to **5 active workspaces**, each with a persistent home (default 5GB, more on request) and created from a **template**: General (noVNC, JupyterLab, terminal, VSCode, Cursor), Selkies, CUDA/PyTorch/TensorFlow, FPGA, and others. Deleting a workspace also removes its storage, so back up anything you need. Each workspace has an SSH key in your user settings for [GitLab](https://gitlab.nrp-nautilus.io/). The [Coder CLI](https://coder.com/docs/install) lets you manage workspaces and SSH in from your machine. More: [Coder on NRP](https://nrp.ai/documentation/userdocs/coder/coder/).

---
## Coder workflow 

We use the **Coder CLI**: install Coder, log in to `coder.nrp-nautilus.io`, create a workspace, install Aider there, and run an agentic coding task (Python sudoku solver + merge request). Run these cells from an environment where Coder is available (e.g. your laptop or a terminal in JupyterHub).

## Install Coder CLI

Install the Coder CLI on your machine (e.g. from JupyterHub terminal or your laptop):

In [ ]:
!curl -L https://coder.com/install.sh | sh

## Log in to NRP Coder

Log in to the NRP Coder instance. This step is **interactive** (browser or token); if running in a JupyterHub cell, you may need to run it in a terminal instead.

In [ ]:
!coder login coder.nrp-nautilus.io

## 0. Create workspace (general-template)

Create a workspace named `test-ubuntu-echo` from the **general-template** with explicit parameters (region Any, 4 CPU cores, 8 GB RAM, no GPU, Ubuntu image). Use your own workspace name if you prefer. **The following steps use the name `test-ubuntu-echo`** in `coder ssh` commands; if you chose a different workspace name, substitute it in the cells below.

In [ ]:
# Create workspace (same parameters as general-template test-ubuntu-echo)
!coder create test-ubuntu-echo -t general-template \
  --parameter '1. Region=Any' \
  --parameter '2. GPUs=0' \
  --parameter '3. CPU Cores=4' \
  --parameter '4. Memory (RAM)=8' \
  --parameter '5. GPU Type=None' \
  --parameter '6. Node=Any' \
  --parameter '7. Image=gitlab-registry.nrp-nautilus.io/nrp/coder-images/ubuntu' \
  --parameter '8. FPGAs=No' \
  -y

## 1. Install Aider in the workspace

Install the **aider-chat** package inside the workspace so we can run an agentic coding task.

In [ ]:
!coder ssh test-ubuntu-echo -- 'pip3 install aider-chat'

## 2. Clone repo into the workspace

Clone the example repo into `~/workspace` and set `OPENAI_API_KEY` and `OPENAI_API_BASE` so Aider uses the NRP LLM endpoint. Adjust the repo URL if using a different project.

In [ ]:
!coder ssh test-ubuntu-echo -- 'export PATH=$HOME/.local/bin:$PATH OPENAI_API_KEY=r4FOpIiGogGdkmpZEiJZj7AreSIUt7d2 OPENAI_API_BASE=https://ellm.nrp-nautilus.io/v1; mkdir -p ~/workspace; cd ~/workspace; if [ -d .git ]; then git pull; else git clone https://gitlab.nrp-nautilus.io/msada/testing-s3-cache.git .; fi'

## 3. Run Aider: create Python sudoku solver and request MR

Set git config and run **Aider** with the NRP LLM endpoint. Aider will create `sudoku.py` (command-line 9×9 solver), add a README section, create branch `aider-sudoku`, commit, and push. The `--message` instructs the agent; Aider may not run `git push`, so we do that in the next step if needed.

In [ ]:
!coder ssh test-ubuntu-echo -- 'export PATH=$HOME/.local/bin:$PATH OPENAI_API_KEY=r4FOpIiGogGdkmpZEiJZj7AreSIUt7d2 OPENAI_API_BASE=https://ellm.nrp-nautilus.io/v1; cd ~/workspace; git config user.email "coder@coder.com"; git config user.name "Coder"; aider --model openai/glm-v --yes --auto-commits --message "Create a Python sudoku solver in sudoku.py (command-line 9x9). Add a README section about it. Then create branch aider-sudoku, commit all changes, push the branch, and open a merge request titled Add Python sudoku solver."'

## 4. Create branch, push, and open merge request

If Aider did not run `git push`, create the branch, push, and open the merge request manually.

In [ ]:
!coder ssh test-ubuntu-echo -- 'cd ~/workspace && git checkout -b aider-sudoku && git push -u origin aider-sudoku -o merge_request.create -o merge_request.title="Add Python sudoku solver"'

## 5. Verify 

Check recent commits, branches, and that `sudoku.py` was created.

In [ ]:
!coder ssh test-ubuntu-echo -- 'cd ~/workspace && git log -3 --oneline && git branch -a && ls -la sudoku.py'

---
## OpenCode and Crush (install and test in this notebook)

This section can be run in this notebook (e.g. in a JupyterHub or `nrp-training-k8s` pod) without a Coder workspace. We install **OpenCode** and **Crush**, point them at the NRP LLM endpoint, and run a short non-interactive test. No `kubectl` or Coder workspace is required.

In [ ]:
# NRP LLM endpoint; set OPENAI_API_KEY in env or below (get a token from https://nrp.ai/llmtoken)
import os
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "r4FOpIiGogGdkmpZEiJZj7AreSIUt7d2"  # tutorial placeholder; prefer env
os.environ["OPENAI_API_BASE"] = "https://ellm.nrp-nautilus.io/v1"
print("OPENAI_API_BASE =", os.environ["OPENAI_API_BASE"])

### Install OpenCode

Install the OpenCode CLI (Node 18+ required). If the install script fails, try `npm install -g opencode-ai` instead.

In [ ]:
!curl -fsSL https://opencode.ai/install | bash

### Install Crush

Install the Crush CLI (Charmbracelet). Requires Node/npm.

In [ ]:
!npm install -g @charmland/crush

### Configure OpenCode for NRP LLM

Point OpenCode at the NRP OpenAI-compatible endpoint via config (base URL + API key). We write `~/.config/opencode/opencode.json` so `opencode run` uses it.

In [ ]:
import json, os
os.makedirs(os.path.expanduser("~/.config/opencode"), exist_ok=True)
cfg = {
    "provider": {
        "openai": {
            "options": {
                "baseURL": "https://ellm.nrp-nautilus.io/v1",
                "apiKey": os.environ["OPENAI_API_KEY"]
            }
        }
    },
    "model": "openai/glm-v"
}
path = os.path.expanduser("~/.config/opencode/opencode.json")
with open(path, "w") as f:
    json.dump(cfg, f, indent=2)
print("Wrote", path)

### Configure Crush for NRP LLM

Write Crush config so it uses the NRP endpoint (OpenAI-compatible with custom base URL).

In [ ]:
import json, os
os.makedirs(os.path.expanduser("~/.config/crush"), exist_ok=True)
cfg = {
    "providers": {
        "nrp-openai": {
            "type": "openai",
            "base_url": "https://ellm.nrp-nautilus.io/v1",
            "api_key": os.environ["OPENAI_API_KEY"],
            "models": ["glm-v"]
        }
    }
}
path = os.path.expanduser("~/.config/crush/crush.json")
with open(path, "w") as f:
    json.dump(cfg, f, indent=2)
print("Wrote", path)

### Test OpenCode

Run a single prompt with OpenCode (non-interactive). Uses the config we wrote above.

In [ ]:
!opencode run "What is 2+2? Reply in one short sentence." --model openai/glm-v

### Test Crush

Run a single prompt with Crush in non-interactive mode (`crush run --prompt ... --yolo`). Uses the config we wrote above.

In [ ]:
!crush run --prompt "What is 2+2? Reply in one short sentence." --model nrp-openai:glm-v --yolo

**Join NRP & hands-on support:** Use these links to [get started with NRP](https://nrp.ai/documentation/userdocs/start/getting-started/) and to [join NRP's Matrix chat](https://nrp.ai/contact/) for hands-on support during the tutorial.

**Related documentation:** [Coder on NRP](https://nrp.ai/documentation/userdocs/coder/coder/) · [NRP-managed LLMs](https://nrp.ai/documentation/userdocs/ai/llm-managed/)

---

## Final Step: Tutorial Survey

Thank you for completing the NAIRR tutorial. Please take a moment to complete the survey—your feedback helps improve future sessions.

**📋 [NAIRR26 Tutorial Survey Slide](NAIRR26%20Tutorial%20Survey%20Slide.pdf)**

*(If the PDF is in this repository root, the link above will open it. Otherwise, use the survey link from the README.)*